In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName('pyspark-pilot')
    .master('spark://spark-master:7077')
    .config('spark.executor.instances', 4)
    .config('spark.executor.cores', 2)
    .config('spark.executor.memory', '2048M')
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/16 19:36:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/16 19:36:11 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
emp_df = spark.read.parquet('/home/spark/data/employee/2')

In [3]:
emp_df.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: string (nullable = true)
 |-- tax: double (nullable = true)



In [4]:
emp_df.show()

+-----------+-------------+-------------+---+------+------+----------+-------+
|employee_id|department_id|         name|age|gender|salary| hire_date|    tax|
+-----------+-------------+-------------+---+------+------+----------+-------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|10000.0|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15| 9000.0|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|11000.0|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30| 9600.0|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|12000.0|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|10400.0|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|14000.0|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|10200.0|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|11600.0|
|        010|          104|     Lisa Lee| 27|Female|

In [5]:
def bonus(salary):
    return int(salary) * 0.1

In [6]:
from pyspark.sql.functions import udf, col

set_bonus = udf(bonus)

emp_new_df = emp_df.withColumn('bonus', set_bonus(col('salary')))

In [7]:
emp_new_df.show()

+-----------+-------------+-------------+---+------+------+----------+-------+------+
|employee_id|department_id|         name|age|gender|salary| hire_date|    tax| bonus|
+-----------+-------------+-------------+---+------+------+----------+-------+------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|10000.0|5000.0|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15| 9000.0|4500.0|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|11000.0|5500.0|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30| 9600.0|4800.0|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|12000.0|6000.0|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|10400.0|5200.0|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|14000.0|7000.0|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|10200.0|5100.0|
|        009|          103|      Tom Tan| 33|  Male| 5

In [8]:
emp_new_df.write.mode('overwrite').parquet('/home/spark/data/employee/3')

In [9]:
spark.stop()